<a href="https://colab.research.google.com/github/asad450/SJSU-Campus-Accessibility-Reporter/blob/main/SJSU_Campus_Accessibility_Reporter_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI4SG Final Prototype: SJSU Campus Accessibility Reporter

This notebook is the prototype file for the final AI4SG GitHub repository. It adapts the class labs into one working system for the **SJSU Campus Accessibility Reporter** project.

The system helps identify, structure, and route accessibility issues around SJSU, such as blocked curb ramps, cracked sidewalks, dark walkways, and inaccessible entrances.

**Before running:**  
1. Save your Gemini API key in Colab Secrets as `GEMINI_API_KEY`.  
2. Turn notebook access ON for that secret.  
3. Run every code cell from top to bottom.  
4. When prompted, upload an accessibility-related image, such as the cracked sidewalk image used in Milestone 2.  
5. Do not clear outputs before uploading this notebook to GitHub.

**AI assistance note:** Generative AI was used to help organize this notebook, adapt lab code structure, and refine explanations. The team reviewed and modified the final content to fit the SJSU Campus Accessibility Reporter project.

## Section 1: Problem and Population

Maria is an SJSU student who uses a wheelchair and needs clear, safe, accessible routes to get to class. The failure point is that accessibility barriers around campus, such as blocked curb ramps, cracked sidewalks, broken lights, or blocked entrances, may be visible or easy to describe, but the report may not turn into structured information that staff can quickly route and act on. This connects to **SDG 10: Reduced Inequalities** because inaccessible spaces create unequal access for students with disabilities, and **SDG 11: Sustainable Cities and Communities** because the goal is to make campus spaces safer, more inclusive, and easier to navigate. Since Milestone 1, the project expanded from focusing mainly on a blocked curb ramp to covering multiple campus accessibility barriers because the cracked-sidewalk image test showed that the system should handle more than one type of mobility issue.

## Section 2: Proposed System

**Workflow**

1. **Input:** A student submits a written accessibility complaint, a photo of the barrier, or both. Examples include a blocked curb ramp, cracked sidewalk, broken light, or blocked entrance.

2. **AI processing:**  
   - **Lab 2, Structured Data Extraction:** Gemini converts the student’s messy written complaint into structured fields: location, accessibility issue, affected user, urgency, responsible department, language, and recommended action.  
   - **Lab 3, Visual Recognition:** Gemini analyzes a photo and identifies the visible accessibility or safety issue, who could be affected, urgency, and who should respond.  
   - **Lab 1, Text Generation:** Gemini drafts a short acknowledgment for the student and a work-order summary for staff.

3. **Output:** A structured report plus a short student-facing response and a staff-facing work-order summary.

4. **Real-world action:** A human reviewer from SJSU Facilities, Accessibility Services, Campus Safety, or City Public Works checks medium/high urgency reports before creating or routing a work order.

The purpose of the AI is to support human triage, not replace it. The system is designed to make reports clearer and faster to review while still keeping a human involved before real-world action happens.

## Section 3: Project Code

This section combines three modified labs into one prototype. The assignment requires at least two labs, but this prototype uses all three because each one supports a different part of the workflow.

### Setup

Run this first. This cell installs the needed packages, imports libraries, and initializes Gemini. The retry helpers are included to reduce problems from temporary API errors or rate limits.

In [ ]:
!pip install -q google-generativeai Pillow

In [ ]:
import google.generativeai as genai
from google.colab import userdata, files
from IPython.display import display
from PIL import Image as PILImage
import json
import time

# Configure Gemini API from Colab Secrets.
# The secret must be named GEMINI_API_KEY and notebook access must be turned on.
api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Missing GEMINI_API_KEY. Add it in Colab Secrets and turn notebook access ON.")

genai.configure(api_key=api_key)
print("Gemini initialized successfully.")


def call_gemini_text(prompt, system_instruction=None, retries=3, wait_seconds=12):
    """Call Gemini for text-only tasks with simple retry handling."""
    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=system_instruction
    ) if system_instruction else genai.GenerativeModel(model_name="gemini-2.5-flash")

    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = model.generate_content(prompt)
            time.sleep(wait_seconds)
            return response.text
        except Exception as error:
            last_error = error
            print(f"Attempt {attempt} failed: {error}")
            if attempt < retries:
                print("Retrying after a short wait...")
                time.sleep(wait_seconds * attempt)

    raise RuntimeError(f"Gemini text call failed after {retries} attempts: {last_error}")


def call_gemini_image(image_path, question, retries=3, wait_seconds=12):
    """Call Gemini with an image plus a question, using simple retry handling."""
    model = genai.GenerativeModel(model_name="gemini-2.5-flash")
    img = PILImage.open(image_path)

    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = model.generate_content([question, img])
            time.sleep(wait_seconds)
            return response.text
        except Exception as error:
            last_error = error
            print(f"Attempt {attempt} failed: {error}")
            if attempt < retries:
                print("Retrying after a short wait...")
                time.sleep(wait_seconds * attempt)

    raise RuntimeError(f"Gemini image call failed after {retries} attempts: {last_error}")


def parse_json_response(raw_text):
    """Clean Gemini output and parse it as JSON."""
    raw = raw_text.strip()

    # Remove markdown code fences if Gemini adds them.
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

    # If extra text appears around JSON, keep only the outer JSON object.
    start = raw.find("{")
    end = raw.rfind("}")
    if start != -1 and end != -1:
        raw = raw[start:end + 1]

    return json.loads(raw)

### Lab 2 Selected: Structured Data Extraction

This code demonstrates the routing part of the proposed system. A student may write a messy complaint, but staff need consistent fields before they can route the issue. The schema forces Gemini to return the same seven fields every time, which makes the output easier to review and send to the correct department.

In [ ]:
# Lab 2: Structured extraction for SJSU accessibility reports

schema_prompt = """
Extract information from this SJSU campus accessibility report.

Return ONLY valid JSON with exactly these seven fields:
{
  "location": string (campus location, building, pathway, entrance, or area described),
  "accessibility_issue": string (blocked ramp, broken sidewalk, dark walkway, inaccessible entrance, broken elevator, etc.),
  "affected_user": string (wheelchair user, student on crutches, visually impaired student, injured student, general pedestrians, etc.),
  "urgency": "LOW" or "MEDIUM" or "HIGH",
  "responsible_department": string (SJSU Facilities Development and Operations, Accessibility Services, Campus Safety, Transportation Services, City Public Works, or Human Review Needed),
  "resident_language": string (language the report was written in),
  "recommended_action": string (the next action staff should take)
}

Urgency guide:
LOW = inconvenience with a clear safe alternate route.
MEDIUM = access is blocked, difficult, or creates a meaningful barrier.
HIGH = immediate safety risk, no safe accessible route, injury risk, or unclear jurisdiction with accessibility impact.

Important routing rule:
If the report contains unclear ownership, unclear location, or uncertainty about whether SJSU or the city is responsible, set responsible_department to "Human Review Needed" and explain what needs to be verified in recommended_action.

No explanation. No markdown. JSON only.
"""


def extract_accessibility_report(message):
    """Convert a messy accessibility complaint into structured routing fields."""
    raw = call_gemini_text(message, system_instruction=schema_prompt)
    return parse_json_response(raw)


test_reports = [
    "The curb ramp near the SJSU library entrance is blocked by construction cones, and a student using a wheelchair had to turn around and find another route.",
    "There is a cracked and uneven sidewalk near the Student Union. A student using crutches almost tripped while walking to class.",
    "The pathway between the parking garage and campus housing is very dark at night because one of the lights is broken. This feels unsafe for students walking alone.",
    "The elevator in one of the campus buildings is out of service, and a student using a wheelchair cannot reach the second floor for class."
]

structured_results = []

for report in test_reports:
    print("INPUT:")
    print(report)
    print("\nOUTPUT:")
    result = extract_accessibility_report(report)
    structured_results.append(result)
    print(json.dumps(result, indent=2, ensure_ascii=False))
    print("-" * 90)

**What this demonstrates:** Lab 2 supports the system by turning a student’s unstructured complaint into structured fields that campus staff could actually route. The important part is not just that Gemini understands the complaint, but that it creates consistent output for location, issue type, affected user, urgency, department, language, and next action. This directly addresses the failure point: accessibility reports are often visible or explainable, but not automatically structured for staff action.

### Lab 1 Selected: Text Generation

This code demonstrates the communication part of the proposed system. Once a report is structured, Gemini drafts two short messages: one acknowledgment to the student and one work-order summary for staff. This matters because students should know their report was understood, while staff need a concise summary they can act on.

In [ ]:
# Lab 1: Text generation for student acknowledgment and staff work-order summary

def draft_messages(structured_report):
    """Generate a student-facing acknowledgment and staff-facing work-order summary."""
    prompt = f"""
You are assisting the SJSU Campus Accessibility Reporter system.

Using the structured report below, write two outputs:

1. A short, respectful acknowledgment message to the student.
   - Be clear and calm.
   - Do not overpromise immediate repair.
   - Mention that the report will be reviewed.

2. A concise work-order summary for campus staff.
   - Include location, issue, affected user, urgency, responsible department, and recommended action.
   - Keep it practical and easy to scan.

Structured report:
{json.dumps(structured_report, indent=2)}

Return the answer with these exact headings:
Student Acknowledgment:
Staff Work-Order Summary:
"""
    return call_gemini_text(prompt)


# Use the broken sidewalk example if available, otherwise use a backup structured report.
if "structured_results" in globals() and len(structured_results) > 1:
    selected_report = structured_results[1]
else:
    selected_report = {
        "location": "near the Student Union",
        "accessibility_issue": "cracked and uneven sidewalk",
        "affected_user": "student on crutches",
        "urgency": "HIGH",
        "responsible_department": "SJSU Facilities Development and Operations",
        "resident_language": "English",
        "recommended_action": "Inspect and repair the cracked and uneven sidewalk to prevent tripping hazards."
    }

print("STRUCTURED REPORT USED:")
print(json.dumps(selected_report, indent=2, ensure_ascii=False))

print("\nGENERATED MESSAGES:")
generated_messages = draft_messages(selected_report)
print(generated_messages)

**What this demonstrates:** Lab 1 supports the system by turning structured output into language that real people can use. The student gets a clear acknowledgment, and staff get a short work-order summary instead of having to read through a messy complaint from scratch. This shows how text generation can support communication without replacing the human reviewer.

### Lab 3 Selected: Visual Recognition

This code demonstrates the image-analysis part of the system. Upload a photo of a cracked sidewalk, blocked ramp, dark walkway, blocked entrance, or similar accessibility issue. The model analyzes what is visible, who could be affected, urgency, and which department should respond.

In [ ]:
# Lab 3: Upload an accessibility-related image.
# Use a cracked sidewalk, blocked ramp, blocked entrance, dark walkway, or similar accessibility issue.

uploaded = files.upload()
if not uploaded:
    raise ValueError("No image uploaded. Please upload an accessibility-related image.")

image_filename = list(uploaded.keys())[0]
img = PILImage.open(image_filename)

print(f"Uploaded: {image_filename}")
print(f"Image size: {img.size[0]}x{img.size[1]} pixels")
display(img)

In [ ]:
# Lab 3: Analyze the uploaded image.

image_questions = [
    (
        "PROBLEM",
        "Describe the accessibility or safety problem visible in this image. Focus on barriers for wheelchair users, students using crutches, visually impaired students, or anyone with limited mobility. Be specific about what you see."
    ),
    (
        "IMPACT",
        "Who could be affected by what is shown in this image? Explain the possible accessibility, safety, or campus mobility impact."
    ),
    (
        "URGENCY",
        "Rate the urgency as LOW, MEDIUM, or HIGH. LOW = minor inconvenience, MEDIUM = access is limited or difficult, HIGH = immediate safety risk or no safe alternate route. Explain your rating briefly."
    ),
    (
        "ACTION",
        "Which campus or city department should respond: SJSU Facilities Development and Operations, Accessibility Services, Campus Safety, Transportation Services, City Public Works, or Human Review Needed? What action should they take?"
    )
]

image_analysis_results = {}

for label, question in image_questions:
    print(f"--- {label} ---")
    answer = call_gemini_image(image_filename, question)
    image_analysis_results[label] = answer
    print(answer)
    print()

**What this demonstrates:** Lab 3 supports the system when the student can show the barrier more easily than explain it. The photo gives visual evidence, while Gemini’s analysis can help staff identify whether the issue is a sidewalk repair, blocked route, lighting/safety problem, or another accessibility concern. A human still needs to review medium/high urgency cases before routing.

## Section 4: Edge Case Elicitation

The goal of this section is to intentionally test a case where the system may fail. The edge case below combines unclear location, unclear jurisdiction, mixed accessibility/safety concerns, and a user outside the assumed majority. This can make the AI sound confident even when it does not have enough information to route the report safely.

In [ ]:
# Edge case: ambiguous jurisdiction + accessibility + safety + uncertain details.
# This is designed to test whether the system admits uncertainty or routes too confidently.

edge_case_report = """
I use a wheelchair and the sidewalk near the bus stop across from the edge of SJSU is cracked and uneven.
At night it is also pretty dark, and one of my wheels got stuck for a second.
I do not know if this belongs to SJSU or the city, and I am not sure what the street name is.
"""

print("EDGE CASE INPUT:")
print(edge_case_report)

edge_result = extract_accessibility_report(edge_case_report)

print("\nAI STRUCTURED OUTPUT:")
print(json.dumps(edge_result, indent=2, ensure_ascii=False))

edge_prompt = f"""
Review this AI-routed accessibility report. Identify one possible failure or uncertainty in the routing.
Then label the result as one of: failure, near-miss, or acceptable.

Original report:
{edge_case_report}

Structured output:
{json.dumps(edge_result, indent=2)}
"""

edge_review = call_gemini_text(edge_prompt)

print("\nEDGE CASE REVIEW:")
print(edge_review)

# Team assessment is included so the ethical finding is explicit even if Gemini's wording varies.
department = str(edge_result.get("responsible_department", "")).lower()
needs_review = "review" in department or "unclear" in department or "unknown" in department

print("\nTEAM ASSESSMENT:")
if needs_review:
    print("Assessment: Acceptable or near-miss. The system recognized uncertainty and avoided fully automatic routing.")
else:
    print("Assessment: Failure. The system chose a specific department even though the user said ownership and location were unclear. This could misroute the report and delay repair for a wheelchair user.")

**Edge case assessment:** The main failure risk is confident routing under uncertainty. If the system chooses one department even though the user does not know whether the issue belongs to SJSU or the city, the report may go to the wrong queue. The real-world consequence is that Maria, or another student using a wheelchair, may keep facing a dangerous path while departments figure out ownership.

**Oversight decision:** Any medium/high urgency accessibility report, or any report with unclear location, ownership, or responsible department, should go to human review before a work order is routed.

**One change and tradeoff:** Add a `jurisdiction_uncertainty` or `human_review_needed` flag to the schema. The tradeoff is slower routing and more staff time, but it reduces the risk of sending urgent accessibility reports to the wrong department.

## Final Notes for GitHub Submission

Before committing this notebook to GitHub:

- Run every code cell from top to bottom.
- Upload the accessibility image when prompted.
- Confirm outputs are visible for Lab 2, Lab 1, Lab 3, and the edge case.
- Do not clear outputs.
- Download this notebook as `.ipynb`.
- Upload it to the GitHub repository as the single prototype file.
- Add screenshots of key outputs to the repository and reference them in the README.

Suggested filename:

`SJSU_Campus_Accessibility_Reporter_Prototype.ipynb`